# MSA Analysis: Influence of Paired vs Unpaired MSAs on Confidence Metrics

This notebook analyzes the influence of paired and unpaired MSAs on confidence metrics for TCR:p:MHC complexes.

**Hypothesis**: Triads and p:MHCs that have higher Neff and interface signal in their paired MSAs have higher confidence metrics.

Based on findings from:
- [AlphaFold2 complex prediction paper](https://www.nature.com/articles/s41467-022-28865-w)
- [AlphaFold3 paper](https://www.nature.com/articles/s41586-024-07487-w#Sec7)

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create results directory
results_dir = Path('../results/msa_analysis')
results_dir.mkdir(parents=True, exist_ok=True)

print("Libraries loaded successfully")

## Load Data

Load the processed MSA metrics data and summary statistics.

In [ ]:
# Load processed data
data_dir = Path('../results/msa_analysis')

# Load parquet files
pmhc_file = data_dir / 'pmhc_msa_metrics.parquet'
triad_file = data_dir / 'triad_msa_metrics.parquet'
summary_file = data_dir / 'msa_analysis_summary.json'

if pmhc_file.exists():
    pmhc_df = pl.read_parquet(pmhc_file)
    print(f"Loaded p:MHC data: {len(pmhc_df)} complexes")
    print(f"Columns: {pmhc_df.columns[:10]}...")  # Show first 10 columns
else:
    print(f"p:MHC data file not found: {pmhc_file}")
    pmhc_df = None

if triad_file.exists():
    triad_df = pl.read_parquet(triad_file)
    print(f"\nLoaded triad data: {len(triad_df)} triads")
    print(f"Columns: {triad_df.columns[:10]}...")  # Show first 10 columns
else:
    print(f"Triad data file not found: {triad_file}")
    triad_df = None

if summary_file.exists():
    with open(summary_file, 'r') as f:
        summary_stats = json.load(f)
    print("\nLoaded summary statistics")
else:
    print(f"Summary file not found: {summary_file}")
    summary_stats = None

## Data Overview

Let's examine the structure and quality of our MSA metrics data.

In [ ]:
def show_data_overview(df, data_type):
    """Display overview of the data."""
    if df is None:
        print(f"No {data_type} data available")
        return
    
    print(f"\n=== {data_type.upper()} Data Overview ===")
    print(f"Shape: {df.shape}")
    
    # Find MSA metric columns
    msa_cols = [col for col in df.columns if any(keyword in col for keyword in ['neff', 'dca_score'])]
    print(f"\nMSA metric columns ({len(msa_cols)}):")
    for col in msa_cols:
        print(f"  - {col}")
    
    # Find confidence metric columns
    conf_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['pae', 'confidence', 'lddt', 'plddt'])]
    print(f"\nConfidence metric columns ({len(conf_cols)}):")
    for col in conf_cols[:10]:  # Show first 10
        print(f"  - {col}")
    if len(conf_cols) > 10:
        print(f"  ... and {len(conf_cols) - 10} more")

show_data_overview(pmhc_df, 'p:MHC')
show_data_overview(triad_df, 'triad')

## MSA Metrics Distribution

Visualize the distribution of Neff values and DCA scores for paired vs unpaired MSAs.

In [ ]:
def plot_msa_distributions(df, data_type, save_plots=True):
    """Plot distributions of MSA metrics."""
    if df is None:
        return
    
    # Convert to pandas for easier plotting
    df_pd = df.to_pandas()
    
    # Find sequence types
    seq_types = []
    if data_type == 'pmhc':
        seq_types = ['mhc_1', 'mhc_2']
    elif data_type == 'triad':
        seq_types = ['mhc_1', 'mhc_2', 'tcr_1', 'tcr_2']
    
    # Plot Neff distributions
    fig, axes = plt.subplots(2, len(seq_types), figsize=(4*len(seq_types), 8))
    if len(seq_types) == 1:
        axes = axes.reshape(-1, 1)
    
    for i, seq_type in enumerate(seq_types):
        paired_col = f"{seq_type}_paired_neff"
        unpaired_col = f"{seq_type}_unpaired_neff"
        
        if paired_col in df_pd.columns and unpaired_col in df_pd.columns:
            # Remove zeros and NaN values for better visualization
            paired_vals = df_pd[paired_col].dropna()
            unpaired_vals = df_pd[unpaired_col].dropna()
            paired_vals = paired_vals[paired_vals > 0]
            unpaired_vals = unpaired_vals[unpaired_vals > 0]
            
            # Histogram
            axes[0, i].hist(unpaired_vals, alpha=0.7, label='Unpaired', bins=30, density=True)
            axes[0, i].hist(paired_vals, alpha=0.7, label='Paired', bins=30, density=True)
            axes[0, i].set_title(f'{seq_type.upper()} Neff Distribution')
            axes[0, i].set_xlabel('Neff')
            axes[0, i].set_ylabel('Density')
            axes[0, i].legend()
            axes[0, i].set_yscale('log')
            
            # Box plot
            data_for_box = [unpaired_vals, paired_vals]
            axes[1, i].boxplot(data_for_box, labels=['Unpaired', 'Paired'])
            axes[1, i].set_title(f'{seq_type.upper()} Neff Box Plot')
            axes[1, i].set_ylabel('Neff')
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(results_dir / f'{data_type}_neff_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot DCA score distributions
    fig, axes = plt.subplots(2, len(seq_types), figsize=(4*len(seq_types), 8))
    if len(seq_types) == 1:
        axes = axes.reshape(-1, 1)
    
    for i, seq_type in enumerate(seq_types):
        paired_col = f"{seq_type}_paired_dca_score"
        unpaired_col = f"{seq_type}_unpaired_dca_score"
        
        if paired_col in df_pd.columns and unpaired_col in df_pd.columns:
            paired_vals = df_pd[paired_col].dropna()
            unpaired_vals = df_pd[unpaired_col].dropna()
            
            # Histogram
            axes[0, i].hist(unpaired_vals, alpha=0.7, label='Unpaired', bins=30, density=True)
            axes[0, i].hist(paired_vals, alpha=0.7, label='Paired', bins=30, density=True)
            axes[0, i].set_title(f'{seq_type.upper()} DCA Score Distribution')
            axes[0, i].set_xlabel('DCA Score')
            axes[0, i].set_ylabel('Density')
            axes[0, i].legend()
            
            # Box plot
            data_for_box = [unpaired_vals, paired_vals]
            axes[1, i].boxplot(data_for_box, labels=['Unpaired', 'Paired'])
            axes[1, i].set_title(f'{seq_type.upper()} DCA Score Box Plot')
            axes[1, i].set_ylabel('DCA Score')
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(results_dir / f'{data_type}_dca_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot distributions
plot_msa_distributions(pmhc_df, 'pmhc')
plot_msa_distributions(triad_df, 'triad')

## Paired vs Unpaired MSA Comparison

Compare the effectiveness of paired vs unpaired MSAs using scatter plots and correlation analysis.

In [ ]:
def plot_paired_vs_unpaired(df, data_type, save_plots=True):
    """Plot paired vs unpaired MSA metrics."""
    if df is None:
        return
    
    df_pd = df.to_pandas()
    
    seq_types = []
    if data_type == 'pmhc':
        seq_types = ['mhc_1', 'mhc_2']
    elif data_type == 'triad':
        seq_types = ['mhc_1', 'mhc_2', 'tcr_1', 'tcr_2']
    
    # Neff comparison
    fig, axes = plt.subplots(2, len(seq_types), figsize=(4*len(seq_types), 8))
    if len(seq_types) == 1:
        axes = axes.reshape(-1, 1)
    
    for i, seq_type in enumerate(seq_types):
        paired_col = f"{seq_type}_paired_neff"
        unpaired_col = f"{seq_type}_unpaired_neff"
        
        if paired_col in df_pd.columns and unpaired_col in df_pd.columns:
            # Remove NaN values
            mask = df_pd[paired_col].notna() & df_pd[unpaired_col].notna()
            paired_vals = df_pd.loc[mask, paired_col]
            unpaired_vals = df_pd.loc[mask, unpaired_col]
            
            if len(paired_vals) > 0:
                # Scatter plot
                axes[0, i].scatter(unpaired_vals, paired_vals, alpha=0.6, s=20)
                axes[0, i].plot([0, max(unpaired_vals.max(), paired_vals.max())], 
                               [0, max(unpaired_vals.max(), paired_vals.max())], 
                               'r--', alpha=0.8, label='y=x')
                axes[0, i].set_xlabel('Unpaired Neff')
                axes[0, i].set_ylabel('Paired Neff')
                axes[0, i].set_title(f'{seq_type.upper()} Neff: Paired vs Unpaired')
                axes[0, i].legend()
                
                # Calculate correlation
                if len(paired_vals) > 1:
                    corr, p_val = pearsonr(unpaired_vals, paired_vals)
                    axes[0, i].text(0.05, 0.95, f'r = {corr:.3f}\np = {p_val:.3e}', 
                                   transform=axes[0, i].transAxes, 
                                   verticalalignment='top',
                                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # Ratio plot
                ratio = paired_vals / (unpaired_vals + 1e-10)
                axes[1, i].hist(ratio, bins=30, alpha=0.7, density=True)
                axes[1, i].axvline(1, color='r', linestyle='--', alpha=0.8, label='Ratio = 1')
                axes[1, i].set_xlabel('Paired/Unpaired Ratio')
                axes[1, i].set_ylabel('Density')
                axes[1, i].set_title(f'{seq_type.upper()} Neff Ratio Distribution')
                axes[1, i].legend()
    
    plt.tight_layout()
    if save_plots:
        plt.savefig(results_dir / f'{data_type}_neff_paired_vs_unpaired.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot comparisons
plot_paired_vs_unpaired(pmhc_df, 'pmhc')
plot_paired_vs_unpaired(triad_df, 'triad')

## Confidence Metric Analysis

Analyze the relationship between MSA metrics and confidence metrics (e.g., PAE scores).

In [ ]:
def analyze_confidence_correlations(df, data_type, save_plots=True):
    """Analyze correlations between MSA metrics and confidence metrics."""
    if df is None:
        return
    
    df_pd = df.to_pandas()
    
    # Find confidence metric columns
    conf_cols = [col for col in df_pd.columns if any(keyword in col.lower() for keyword in ['pae', 'confidence', 'lddt', 'plddt'])]
    
    # Find MSA metric columns
    msa_cols = [col for col in df_pd.columns if any(keyword in col for keyword in ['neff', 'dca_score'])]
    
    if not conf_cols or not msa_cols:
        print(f"No confidence or MSA metrics found in {data_type} data")
        print(f"Confidence cols: {conf_cols[:5]}")
        print(f"MSA cols: {msa_cols[:5]}")
        return
    
    # Calculate correlation matrix
    correlations = {}
    
    for conf_col in conf_cols[:5]:  # Limit to first 5 confidence metrics
        for msa_col in msa_cols:
            # Remove NaN values
            mask = df_pd[conf_col].notna() & df_pd[msa_col].notna()
            conf_vals = df_pd.loc[mask, conf_col]
            msa_vals = df_pd.loc[mask, msa_col]
            
            if len(conf_vals) > 10:  # Need sufficient data points
                corr_pearson, p_val = pearsonr(conf_vals, msa_vals)
                corr_spearman, _ = spearmanr(conf_vals, msa_vals)
                
                correlations[f"{conf_col}_vs_{msa_col}"] = {
                    'pearson': corr_pearson,
                    'spearman': corr_spearman,
                    'p_value': p_val,
                    'n_samples': len(conf_vals)
                }
    
    # Plot top correlations
    if correlations:
        # Sort by absolute Pearson correlation
        sorted_corrs = sorted(correlations.items(), 
                             key=lambda x: abs(x[1]['pearson']), 
                             reverse=True)
        
        # Plot top 6 correlations
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, (key, corr_data) in enumerate(sorted_corrs[:6]):
            conf_col, msa_col = key.split('_vs_')
            
            # Get data
            mask = df_pd[conf_col].notna() & df_pd[msa_col].notna()
            conf_vals = df_pd.loc[mask, conf_col]
            msa_vals = df_pd.loc[mask, msa_col]
            
            # Create scatter plot
            axes[idx].scatter(msa_vals, conf_vals, alpha=0.6, s=10)
            
            # Add trend line
            z = np.polyfit(msa_vals, conf_vals, 1)
            p = np.poly1d(z)
            axes[idx].plot(msa_vals.sort_values(), p(msa_vals.sort_values()), "r--", alpha=0.8)
            
            axes[idx].set_xlabel(msa_col.replace('_', ' ').title())
            axes[idx].set_ylabel(conf_col.replace('_', ' ').title())
            axes[idx].set_title(f'r = {corr_data["pearson"]:.3f}, p = {corr_data["p_value"]:.2e}')
        
        plt.tight_layout()
        if save_plots:
            plt.savefig(results_dir / f'{data_type}_confidence_correlations.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print correlation summary
        print(f"\n=== Top Correlations for {data_type.upper()} ===")
        for key, corr_data in sorted_corrs[:10]:
            print(f"{key}: r = {corr_data['pearson']:.3f}, p = {corr_data['p_value']:.2e}, n = {corr_data['n_samples']}")
    
    return correlations

# Analyze correlations
pmhc_correlations = analyze_confidence_correlations(pmhc_df, 'pmhc')
triad_correlations = analyze_confidence_correlations(triad_df, 'triad')

## Summary Statistics

Display and visualize the summary statistics from our analysis.

In [ ]:
if summary_stats:
    print("=== SUMMARY STATISTICS ===")
    
    # p:MHC Analysis
    pmhc_stats = summary_stats.get('pmhc_analysis', {})
    print(f"\np:MHC Complexes Analyzed: {pmhc_stats.get('n_complexes', 0)}")
    
    # Show paired vs unpaired comparisons
    pmhc_comparisons = pmhc_stats.get('paired_vs_unpaired_comparison', {})
    print("\np:MHC Paired vs Unpaired Neff Comparisons:")
    for key, value in pmhc_comparisons.items():
        if 'neff_comparison' in key:
            seq_type = key.replace('_neff_comparison', '')
            mean_ratio = value.get('mean_ratio_paired_to_unpaired', 0)
            paired_higher = value.get('paired_higher_count', 0)
            unpaired_higher = value.get('unpaired_higher_count', 0)
            print(f"  {seq_type}: Mean paired/unpaired ratio = {mean_ratio:.2f}")
            print(f"    Paired higher: {paired_higher}, Unpaired higher: {unpaired_higher}")
    
    # Triad Analysis
    triad_stats = summary_stats.get('triad_analysis', {})
    print(f"\nTriads Analyzed: {triad_stats.get('n_triads', 0)}")
    
    triad_comparisons = triad_stats.get('paired_vs_unpaired_comparison', {})
    print("\nTriad Paired vs Unpaired Neff Comparisons:")
    for key, value in triad_comparisons.items():
        if 'neff_comparison' in key:
            seq_type = key.replace('_neff_comparison', '')
            mean_ratio = value.get('mean_ratio_paired_to_unpaired', 0)
            paired_higher = value.get('paired_higher_count', 0)
            unpaired_higher = value.get('unpaired_higher_count', 0)
            print(f"  {seq_type}: Mean paired/unpaired ratio = {mean_ratio:.2f}")
            print(f"    Paired higher: {paired_higher}, Unpaired higher: {unpaired_higher}")
else:
    print("Summary statistics not available. Run the pipeline first.")

## Conclusions and Next Steps

Based on our analysis, we can draw the following conclusions about the influence of paired vs unpaired MSAs on confidence metrics:

### Key Findings:

1. **Neff Distribution**: [To be filled based on results]
2. **Paired vs Unpaired Performance**: [To be filled based on results]
3. **Correlation with Confidence Metrics**: [To be filled based on results]

### Implications:

- [To be filled based on results]

### Future Work:

1. Implement full DCA analysis using more sophisticated methods
2. Use structural data to define interface positions more accurately
3. Analyze correlation with experimental binding data if available
4. Extend analysis to other confidence metrics

### References:

- [AlphaFold2 complex prediction](https://www.nature.com/articles/s41467-022-28865-w)
- [AlphaFold3 MSA analysis](https://www.nature.com/articles/s41586-024-07487-w#Sec7)

In [ ]:
print("Analysis completed!")
print(f"Results saved in: {results_dir}")
print("\nFiles created:")
for file_path in results_dir.glob('*.png'):
    print(f"  - {file_path.name}")
for file_path in results_dir.glob('*.json'):
    print(f"  - {file_path.name}")